# Chapter 7 - Tune Cell

## 7.1 Introduction

In order to adjust the tune of the ring, the phase advance per FODO cell in one of the straight sections can be adjusted, while ensuring it is still matching to the periodic solution in each of the surrounding arcs. We will put this **tune cell** in the 2 o'clock straight section and modify the lines and lattice elements by appending `_2` to their names.

Required for the optimization is:

1. Set the phase advances in the `FODOSSF_2`/`FODOSSR_2` straight section for the desired tunes
   \[
   Q_x = 54.08,\qquad Q_y = 54.14.
   \]
2. Match from the dispersion suppressor to the new periodic betas and alphas in the center straight section.
3. Match the center straight section to the dispersion creator. This side is already defined by mirror symmetry.

The six independent knobs are the four quadrupole families in the forward matching section,
`QFF2_2`, `QDF2_2`, `QFF3_2`, and `QDF3_2`, together with the two straight-section families, `QFSS_2` and `QDSS_2`.

<div align="center"><img src="assets/chapter7_tune_cell_layout.png" width="900"></div>

<p align="center"><em>Figure 1: The 2 o'clock tune cell. The four right-side matching families are fixed by mirror symmetry.</em></p>

## 7.2 Six Knobs and Six Independent Constraints

Although the straight section contains many physical quadrupoles, every `QFSS_2` magnet shares one strength and every `QDSS_2` magnet shares another. Therefore, the straight contributes only two independent knobs.

The optimization uses:

| Independent variables | Purpose |
|---|---|
| `QFF2_2`, `QDF2_2`, `QFF3_2`, `QDF3_2` | Match the arc-side optics to the modified straight section |
| `QFSS_2`, `QDSS_2` | Change the periodic straight-cell optics and phase advances |

At the center of the straight, the fourth and fifth focusing quadrupoles are one FODO period apart. Requiring their two betas and two alphas to be equal gives four independent periodicity constraints. The two desired full-ring tunes provide two more:

\[
\underbrace{\Delta\beta_1,\Delta\alpha_1,\Delta\beta_2,\Delta\alpha_2}_{4\ \text{periodicity constraints}}
+
\underbrace{Q_x-54.08,\ Q_y-54.14}_{2\ \text{tune constraints}}
= 0.
\]

Once the left side is matched and the center is periodic, the right side matches to the dispersion creator automatically because its elements and target optics are the mirror image of the left side.

In [1]:
using SciBmad
using GTPSA
using LinearAlgebra
using Printf

tutorial_root = @__DIR__
include(joinpath(tutorial_root, "lattices", "chapter_5", "chapter5_ring_definition.jl"))
include(joinpath(tutorial_root, "lattices", "chapter_6", "chapter6_IR_solution.jl"))

const C5 = Chapter5Ring
const TARGET_TUNES = [54.08, 54.14]
const TUNE_DESCRIPTOR = Descriptor(6, 2, 6, 1)
const DK_TUNE = params(TUNE_DESCRIPTOR)

6-element Vector{TPS64{GTPSA.Dynamic}}:
Descriptor(NV=6, MO=2, NP=6, PO=1)
INDEX  COEFFICIENT             ORDER   EXPONENTS
----------------------------------------------------------
  1:  1.0000000000000000E+000    1     0 0  0 0  0 0  7^1
----------------------------------------------------------
  2:  1.0000000000000000E+000    1     0 0  0 0  0 0  8^1
----------------------------------------------------------
  3:  1.0000000000000000E+000    1     0 0  0 0  0 0  9^1
----------------------------------------------------------
  4:  1.0000000000000000E+000    1     0 0  0 0  0 0  10^1
----------------------------------------------------------
  5:  1.0000000000000000E+000    1     0 0  0 0  0 0  11^1
----------------------------------------------------------
  6:  1.0000000000000000E+000    1     0 0  0 0  0 0  12^1


### Start from the Chapter 6 ring

The initial strengths are the ordinary Chapter 5 matching and straight-section strengths. At this point the `_2` region is physically identical to the original straight; only its names are new. These six starting values will become the optimization variables.

In [2]:
# Variable order: QFF2_2, QDF2_2, QFF3_2, QDF3_2, QFSS_2, QDSS_2.
K_START = [
    C5.quad_strength[:QFF2],
    C5.quad_strength[:QDF2],
    C5.quad_strength[:QFF3],
    C5.quad_strength[:QDF3],
    C5.quad_strength[:QFSS],
    C5.quad_strength[:QDSS],
]

function special_quad_strength(k; knobs=nothing)
    strength(i) = isnothing(knobs) ? k[i] : k[i] + knobs[i]
    return Dict(
        :QFF2_2 => strength(1), :QDF2_2 => strength(2),
        :QFF3_2 => strength(3), :QDF3_2 => strength(4),
        :QFSS_2 => strength(5), :QDSS_2 => strength(6),
    )
end

function make_tune_element(name::Symbol, k; knobs=nothing)
    strengths = special_quad_strength(k; knobs=knobs)
    haskey(strengths, name) &&
        return Quadrupole(name=String(name), L=C5.L_quad, Kn1=strengths[name])
    return C5.make_element(name)
end

make_tune_elements(line, k; knobs=nothing) = [make_tune_element(name, k; knobs=knobs) for name in line]

@show K_START

K_START = [0.35627203960957676, -0.34660350952645963, 0.3788109448047574, -0.3516264952123834, 0.351957452649287, -0.351957452649287]


6-element Vector{Float64}:
  0.35627203960957676
 -0.34660350952645963
  0.3788109448047574
 -0.3516264952123834
  0.351957452649287
 -0.351957452649287

### Define the special straight and its mirror-symmetric matching sections

Every occurrence of `QFSS_2` uses `k[5]`, and every occurrence of `QDSS_2` uses `k[6]`. The right-side line uses the same four matching strengths as the left side, but in mirror-reversed order.

In [3]:
FODOSSF_2 = [:QFSS_2, :D1, :DB, :D2, :QDSS_2, :D1, :DB, :D2]
FODOSSR_2 = [:QFSS_2, :D2, :DB, :D1, :QDSS_2, :D2, :DB, :D1]

ARC_TO_SSF_2 = [
    :QF, :D1, :BH, :D2, :QD, :D1, :BH, :D2,
    :QFF1, :D1, :BH, :D2, :QDF1, :D1, :BH, :D2,
    :QFF2_2, :D1, :DB, :D2, :QDF2_2, :D1, :DB, :D2,
    :QFF3_2, :D1, :DB, :D2, :QDF3_2, :D1, :DB, :D2,
]

SS_TO_ARCR_2 = [
    :QFSS_2, :D2, :DB, :D1, :QDF3_2, :D2, :DB, :D1,
    :QFF3_2, :D2, :DB, :D1, :QDF2_2, :D2, :DB, :D1,
    :QFF2_2, :D2, :BH, :D1, :QDF1, :D2, :BH, :D1,
    :QFF1, :D2, :BH, :D1, :QD, :D2, :BH, :D1,
]

32-element Vector{Symbol}:
 :QFSS_2
 :D2
 :DB
 :D1
 :QDF3_2
 :D2
 :DB
 :D1
 :QFF3_2
 :D2
 :DB
 :D1
 :QDF2_2
 ⋮
 :QDF1
 :D2
 :BH
 :D1
 :QFF1
 :D2
 :BH
 :D1
 :QD
 :D2
 :BH
 :D1

### Preserve the Chapter 6 low-beta interaction region

Chapter 7 follows the Chapter 6 lattice, so the optimized IR at 6 o'clock remains in the ring while the new tune cell is inserted at 2 o'clock.

In [4]:
function build_IPF_elements()
    return [
        Quadrupole(name="QEF1", L=0.5, Kn1=K_QEF1),
        C5.make_element(:D1), C5.make_element(:DB), C5.make_element(:D2),
        Quadrupole(name="QEF2", L=0.5, Kn1=K_QEF2),
        C5.make_element(:D1), C5.make_element(:DB), C5.make_element(:D2),
        Drift(name="DEF1", L=20.46),
        Quadrupole(name="QEF3", L=1.6, Kn1=K_QEF3),
        Drift(name="DEF2", L=3.76),
        Quadrupole(name="QEF4", L=1.2, Kn1=K_QEF4),
        Drift(name="DEF3", L=5.8),
        Marker(name="IP6"),
    ]
end

function build_IPR_elements()
    return [
        Drift(name="DER3", L=5.3),
        Quadrupole(name="QER4", L=1.8, Kn1=K_QER4),
        Drift(name="DER2", L=0.5),
        Quadrupole(name="QER3", L=1.4, Kn1=K_QER3),
        Drift(name="DER1", L=23.82),
        Quadrupole(name="QER2", L=0.5, Kn1=K_QER2),
        C5.make_element(:D2), C5.make_element(:DB), C5.make_element(:D1),
        Quadrupole(name="QER1", L=0.5, Kn1=K_QER1),
        C5.make_element(:D2), C5.make_element(:DB), C5.make_element(:D1),
    ]
end

build_IPR_elements (generic function with 1 method)

### Build the complete ring for any trial vector

The tune cell lies across the boundary between `SEXTANT1` and `SEXTANT3`. Only that 2 o'clock straight and its adjacent matching sections receive `_2` elements. The remaining sextants, including the Chapter 6 IR, are unchanged.

In [5]:
function build_ring_with_tune_cell(k; knobs=nothing)
    sextant1_tune = vcat(
        C5.make_elements(C5.repeat_line(C5.FODOSSF, 4)),
        C5.make_elements(C5.SS_TO_ARCF),
        C5.make_elements(C5.repeat_line(C5.FODOAF, 20)),
        make_tune_elements(ARC_TO_SSF_2, k; knobs=knobs),
        make_tune_elements(C5.repeat_line(FODOSSF_2, 4), k; knobs=knobs),
    )

    sextant3_tune = vcat(
        make_tune_elements(C5.repeat_line(FODOSSR_2, 4), k; knobs=knobs),
        make_tune_elements(SS_TO_ARCR_2, k; knobs=knobs),
        C5.make_elements(C5.repeat_line(C5.FODOAR, 20)),
        C5.make_elements(C5.ARC_TO_SSR),
        C5.make_elements(C5.repeat_line(C5.FODOSSR, 4)),
    )

    sextant5_ir = vcat(
        C5.make_elements(C5.repeat_line(C5.FODOSSF, 4)),
        C5.make_elements(C5.SS_TO_ARCF),
        C5.make_elements(C5.repeat_line(C5.FODOAF, 20)),
        C5.make_elements(C5.ARC_TO_SSF),
        C5.make_elements(C5.FODOSSF),
        build_IPF_elements(),
    )

    sextant7_ir = vcat(
        build_IPR_elements(),
        C5.make_elements(C5.FODOSSR),
        C5.make_elements(C5.SS_TO_ARCR),
        C5.make_elements(C5.repeat_line(C5.FODOAR, 20)),
        C5.make_elements(C5.ARC_TO_SSR),
        C5.make_elements(C5.repeat_line(C5.FODOSSR, 4)),
    )

    elements = vcat(
        sextant1_tune, sextant3_tune, sextant5_ir, sextant7_ir,
        C5.make_elements(C5.SEXTANT9), C5.make_elements(C5.SEXTANT11),
    )
    ring = Beamline(elements; species_ref=C5.species_ref, E_ref=C5.E_ref)
    return ring, elements
end

ring_start, elements_start = build_ring_with_tune_cell(K_START)
@show length(elements_start) count(ele -> ele.name == "QFSS_2", elements_start)

length(elements_start) = 1707
count((ele->begin
            #= In[5]:45 =#
            ele.name == "QFSS_2"
        end), elements_start) = 9


9

### Read the tunes and impose periodicity at the center

SciBmad stores accumulated phase in turns, so the final values `phi_1[end]` and `phi_2[end]` are directly the full-ring tunes.

There are nine `QFSS_2` elements in the complete tune-cell region: four in `FODOSSF_2`, four in `FODOSSR_2`, and one at the beginning of the mirror matching line. Following the PDF, the fourth and fifth are one FODO period apart at the center. Equal betas and alphas at those two positions enforce the new periodic straight-cell solution.

In [6]:
optics_table(tw) = hasproperty(tw, :table) ? tw.table : tw

zero6 = [0, 0, 0, 0, 0, 0]

function tps_const(x)
    try
        return x[zero6]
    catch
        return x
    end
end

parameter_gradient(x) = GTPSA.gradient(x, include_params=true)[7:end]

function straight_cell_phase_advances(k)
    # Measure one center FODO period inside the complete stable ring.
    ring, elements = build_ring_with_tune_cell(k)
    table = optics_table(twiss(ring))
    qf_indices = findall(ele -> ele.name == "QFSS_2", elements)
    row4 = qf_indices[4] + 1
    row5 = qf_indices[5] + 1
    return 360 .* tps_const.([
        table.phi_1[row5] - table.phi_1[row4],
        table.phi_2[row5] - table.phi_2[row4],
    ])
end

function tune_cell_metrics(k; knobs=nothing, descriptor=nothing, constants=true)
    ring, elements = build_ring_with_tune_cell(k; knobs=knobs)
    tw = isnothing(descriptor) ? twiss(ring) : twiss(ring, GTPSA_descriptor=descriptor)
    table = optics_table(tw)

    qf_indices = findall(ele -> ele.name == "QFSS_2", elements)
    row4 = qf_indices[4] + 1  # Twiss table row after QFSS_2##4.
    row5 = qf_indices[5] + 1  # Twiss table row after QFSS_2##5.

    periodicity = [
        (table.beta_1[row4] - table.beta_1[row5]) / table.beta_1[row5],
        table.alpha_1[row4] - table.alpha_1[row5],
        (table.beta_2[row4] - table.beta_2[row5]) / table.beta_2[row5],
        table.alpha_2[row4] - table.alpha_2[row5],
    ]
    tunes = [table.phi_1[end], table.phi_2[end]]

    if constants
        periodicity = tps_const.(periodicity)
        tunes = tps_const.(tunes)
    end

    return (periodicity=periodicity, tunes=tunes)
end

function tune_cell_residual(k)
    metrics = tune_cell_metrics(k)
    return vcat(metrics.periodicity, metrics.tunes - TARGET_TUNES)
end

tune_cell_residual (generic function with 1 method)

### Evaluate the unmodified starting lattice

Because the initial `_2` strengths equal the ordinary straight-section strengths, the center is already periodic and the straight FODO cell begins near \(90^\circ\) phase advance in both planes. Its full-ring tunes, however, are not the requested Chapter 7 working point.

In [7]:
initial = tune_cell_metrics(K_START)
@printf("Initial Qx = %.12f, Qy = %.12f\n", initial.tunes...)
@printf("Initial straight-cell phase advances = %.9f deg, %.9f deg\n",
        straight_cell_phase_advances(K_START)...)
println("Initial periodicity residual = ", initial.periodicity)
println("Initial complete residual    = ", tune_cell_residual(K_START))

Initial Qx = 54.512285929095, Qy = 53.163888260647
Initial straight-cell phase advances = 90.002836096 deg, 90.000500182 deg
Initial periodicity residual = [0.0006671376910707568, 0.0017012559329296906, 6.954655725757075e-6, 1.408939096053663e-5]
Initial complete residual    = [0.0006671376910707568, 0.0017012559329296906, 6.954655725757075e-6, 1.408939096053663e-5, 0.4322859290950021, -0.9761117393525183]


## 7.3 Optimize the Six Quadrupole Families

The outer optimization Jacobian is calculated with GTPSA descriptor parameters. The six quadrupole-family strength changes are stored as descriptor parameters, so each residual evaluation performs one full periodic Twiss calculation of the ring and the residual derivatives are read directly from the parameter gradients. A Levenberg-Marquardt-style damping term reduces the step whenever a direct Gauss-Newton step would increase the merit function.

The residual vector has exactly six components:

```text
[normalized Delta beta_1, Delta alpha_1,
 normalized Delta beta_2, Delta alpha_2,
 Qx - 54.08, Qy - 54.14]
```

In [8]:
function tune_cell_residual_with_knobs(k)
    metrics = tune_cell_metrics(
        k;
        knobs=DK_TUNE,
        descriptor=TUNE_DESCRIPTOR,
        constants=false,
    )
    return vcat(metrics.periodicity, metrics.tunes .- TARGET_TUNES)
end

function tune_cell_residual_jacobian(k)
    residual = tune_cell_residual_with_knobs(k)
    return vcat((parameter_gradient(r)' for r in residual)...)
end

function damped_least_squares(
    f, x0; jacobian=tune_cell_residual_jacobian, maxiter=60, tol=1e-11, lambda0=1e-3,
)
    x = copy(x0)
    lambda = lambda0

    for iter in 1:maxiter
        r = f(x)
        merit_now = sum(abs2, r)
        J = jacobian(x)
        step = -(J' * J + lambda * I) \ (J' * r)
        trial = x + step
        merit_trial = sum(abs2, f(trial))

        @printf("iter %2d  merit = %.6e  step = %.3e  lambda = %.3e\n",
                iter, merit_now, norm(step), lambda)

        if merit_trial < merit_now
            x = trial
            lambda /= 10
        else
            lambda *= 10
        end

        norm(r) < tol && break
        norm(step) < tol && break
    end
    return x
end

damped_least_squares (generic function with 1 method)

In [10]:
K_TUNE = damped_least_squares(tune_cell_residual, K_START)

iter  1  merit = 1.139669e+00  step = 1.085e-01  lambda = 1.000e-03
iter  2  merit = 5.244751e-01  step = 2.591e-01  lambda = 1.000e-04
iter  3  merit = 5.244751e-01  step = 2.425e-01  lambda = 1.000e-03
iter  4  merit = 5.244751e-01  step = 1.480e-01  lambda = 1.000e-02
iter  5  merit = 5.244751e-01  step = 3.099e-02  lambda = 1.000e-01
iter  6  merit = 6.463304e-03  step = 2.581e-03  lambda = 1.000e-02
iter  7  merit = 2.997541e-06  step = 1.363e-05  lambda = 1.000e-03
iter  8  merit = 1.782500e-10  step = 2.798e-06  lambda = 1.000e-04
iter  9  merit = 1.782339e-10  step = 2.788e-05  lambda = 1.000e-05
iter 10  merit = 1.782186e-10  step = 2.789e-04  lambda = 1.000e-06
iter 11  merit = 1.782186e-10  step = 2.790e-05  lambda = 1.000e-05
iter 12  merit = 1.782031e-10  step = 2.789e-04  lambda = 1.000e-06
iter 13  merit = 1.782031e-10  step = 2.790e-05  lambda = 1.000e-05
iter 14  merit = 1.781875e-10  step = 2.789e-04  lambda = 1.000e-06
iter 15  merit = 1.781875e-10  step = 2.791e-05 

6-element Vector{Float64}:
  0.3336950220855002
 -0.3369687801113691
  0.3522826588843806
 -0.39749522454809927
  0.3370696898025157
 -0.44234837004133726

## 7.4 Final Checks

The final check reports the six optimized family strengths, the full-ring tunes, the modified phase advance of one straight FODO cell, and the center-periodicity residual. Notice that `QFSS_2` and `QDSS_2` no longer form an equal-and-opposite pair; two independent strengths are needed to control the two phase advances.

In [11]:
final = tune_cell_metrics(K_TUNE)
names = ["K_QFF2_2", "K_QDF2_2", "K_QFF3_2", "K_QDF3_2", "K_QFSS_2", "K_QDSS_2"]

println("Optimized tune-cell strengths:")
for (name, value) in zip(names, K_TUNE)
    @printf("  %-10s = %+.15f\n", name, value)
end

@printf("\nFinal Qx = %.12f  target = %.12f\n", final.tunes[1], TARGET_TUNES[1])
@printf("Final Qy = %.12f  target = %.12f\n", final.tunes[2], TARGET_TUNES[2])
@printf("Final straight-cell phase advances = %.9f deg, %.9f deg\n",
        straight_cell_phase_advances(K_TUNE)...)
println("Final periodicity residual = ", final.periodicity)
println("Final complete residual    = ", tune_cell_residual(K_TUNE))

Optimized tune-cell strengths:
  K_QFF2_2   = +0.333695022085500
  K_QDF2_2   = -0.336968780111369
  K_QFF3_2   = +0.352282658884381
  K_QDF3_2   = -0.397495224548099
  K_QFSS_2   = +0.337069689802516
  K_QDSS_2   = -0.442348370041337

Final Qx = 54.079999999783  target = 54.080000000000
Final Qy = 54.140000002568  target = 54.140000000000
Final straight-cell phase advances = 77.003194868 deg, 128.936377792 deg
Final periodicity residual = [4.335552209322577e-6, -1.4961239003241644e-6, 7.928968254201607e-6, 9.691510656939517e-6]
Final complete residual    = [4.335552209322577e-6, -1.4961239003241644e-6, 7.928968254201607e-6, 9.691510656939517e-6, -2.170850166294258e-10, 2.568185664131306e-9]


## 7.5 Exercises

1. Change the target tunes and determine how far the tune cell can move the working point before the optimization encounters an unstable lattice.
2. Remove the four center-periodicity residuals and optimize only the tunes. Compare the resulting straight-section betas and alphas at `QFSS_2##4` and `QFSS_2##5`.
3. Give the right-side matching families independent strengths instead of enforcing mirror symmetry. Explain why this introduces four additional variables and requires additional matching constraints.
4. Compare the final phase advances of `FODOSSF_2` with the original \(90^\circ\) straight FODO cell.